In [1]:
from google.colab import files
uploaded = files.upload()

Saving combined_data(2025-3-1-2026-2-28) by channel and user.csv to combined_data(2025-3-1-2026-2-28) by channel and user.csv


In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json
import os
import warnings
warnings.filterwarnings("ignore")

# ── Frammer Theme ─────────────────────────────────────────────────────────────
FRAMMER_COLORS = ["#7A0000", "#CC0000", "#FF4649", "#FF9A9A", "#FFD6D6", "#FFFFFF"]

FRAMMER_LAYOUT = dict(
    paper_bgcolor="#0D0D0D",
    plot_bgcolor ="#0D0D0D",
    font         =dict(color="#FFFFFF", family="DejaVu Sans"),
    title_font   =dict(color="#FF3131", size=14),
    xaxis        =dict(gridcolor="#2A2A2A", linecolor="#444444"),
    yaxis        =dict(gridcolor="#2A2A2A", linecolor="#444444"),
)

# ── Folder to save JSON exports ───────────────────────────────────────────────
os.makedirs("frammer_charts", exist_ok=True)

def save_chart(fig, filename):
    """Save Plotly chart as JSON for React frontend"""
    filepath = f"frammer_charts/{filename}.json"
    with open(filepath, "w") as f:
        f.write(fig.to_json())
    print(f"✅ Saved → {filepath}")
    fig.show()

print("✅ Plotly imports done | Frammer theme ready | Export folder created")

✅ Plotly imports done | Frammer theme ready | Export folder created


In [4]:
df = pd.read_csv("combined_data(2025-3-1-2026-2-28) by channel and user.csv")
df.info()

#To find the missing values
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236 entries, 0 to 235
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Channel                        236 non-null    object
 1   User                           236 non-null    object
 2   Uploaded Count                 236 non-null    int64 
 3   Created Count                  236 non-null    int64 
 4   Published Count                236 non-null    int64 
 5   Uploaded Duration (hh:mm:ss)   236 non-null    object
 6   Created Duration (hh:mm:ss)    236 non-null    object
 7   Published Duration (hh:mm:ss)  236 non-null    object
dtypes: int64(3), object(5)
memory usage: 14.9+ KB


,0
Channel,0
User,0
Uploaded Count,0
Created Count,0
Published Count,0
Uploaded Duration (hh:mm:ss),0
Created Duration (hh:mm:ss),0
Published Duration (hh:mm:ss),0


In [5]:
df = df.copy()

# IMPORTANT: To remove Trailing Space
df["Channel"] = df["Channel"].astype(str).str.strip()
df["User"]    = df["User"].astype(str).str.strip()

# To Convert hh:mm:ss duration columns → total seconds
def hhmmss_to_seconds(series):
    def _parse(val):
        try:
            h, m, s = str(val).strip().split(":")
            return int(h) * 3600 + int(m) * 60 + int(s)
        except:
            return 0
    return series.apply(_parse)

df["uploaded_dur_sec"]  = hhmmss_to_seconds(df["Uploaded Duration (hh:mm:ss)"])
df["created_dur_sec"]   = hhmmss_to_seconds(df["Created Duration (hh:mm:ss)"])
df["published_dur_sec"] = hhmmss_to_seconds(df["Published Duration (hh:mm:ss)"])

# Rename
df.rename(columns={
    "Uploaded Count"  : "uploaded_cnt",
    "Created Count"   : "created_cnt",
    "Published Count" : "published_cnt",
}, inplace=True)

# Metric Definitions
# Publish Rate %  (avoid division by zero)
df["publish_rate_pct"] = np.where(
    df["created_cnt"] > 0,
    (df["published_cnt"] / df["created_cnt"] * 100).round(2),
    0.0
)

# Amplification Ratio — how many clips created per upload
df["amplification_ratio"] = np.where(
    df["uploaded_cnt"] > 0,
    (df["created_cnt"] / df["uploaded_cnt"]).round(2),
    0.0
)

# Remove internal or test accounts
TEST_KEYWORDS = ["test", "deleteme", "qa-", "frammer.com",
                 "moolya.com", "testaing.com", "theveritycorp.com"]

df["is_internal"] = df["User"].str.lower().apply(
    lambda u: any(kw in u for kw in TEST_KEYWORDS)
)

df_clean = df[df["is_internal"] == False].reset_index(drop=True)

print(f"✅ Preprocessing done")
print(f"   Rows: {len(df)} | Channels: {df['Channel'].nunique()} | Users: {df['User'].nunique()}")
print(f"   Internal accounts flagged: {df['is_internal'].sum()}")
print(f"   Rows after removing internal accounts: {len(df_clean)}")

✅ Preprocessing done
   Rows: 236 | Channels: 18 | Users: 45
   Internal accounts flagged: 56
   Rows after removing internal accounts: 180


In [6]:
# ── KPI-06 Calculation ────────────────────────────────────────────────────────
kpi06 = (
    df_clean.groupby("Channel")
    .agg(total_published=("published_cnt", "sum"),
         total_created  =("created_cnt",   "sum"))
    .assign(publish_rate_pct=lambda x:
            (x["total_published"] / x["total_created"] * 100).round(2))
    .reset_index()
    .sort_values("publish_rate_pct", ascending=False)
)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Bar(
    x=kpi06["Channel"],
    y=kpi06["publish_rate_pct"],
    marker_color=FRAMMER_COLORS[2],
    marker_line_color="#0D0D0D",
    marker_line_width=1.5,
    text=kpi06["publish_rate_pct"].apply(lambda x: f"{x:.1f}%"),
    textposition="outside",
    textfont=dict(color="#FFFFFF", size=10),
    hovertemplate="<b>Channel %{x}</b><br>Publish Rate: %{y:.1f}%<extra></extra>",
))

# 3% Benchmark line
fig.add_hline(
    y=3,
    line_dash="dash",
    line_color=FRAMMER_COLORS[4],
    line_width=1.5,
    annotation_text="3% Benchmark",
    annotation_font_color=FRAMMER_COLORS[4],
    annotation_position="top right"
)

fig.update_layout(
    **FRAMMER_LAYOUT,
    title="KPI-06 · Publish Rate % by Channel<br><sup>Published ÷ Created × 100</sup>",
    xaxis_title="Channel",
    yaxis_title="Publish Rate (%)",
    height=500,
    width=900,
)

save_chart(fig, "kpi06_publish_rate_by_channel")

✅ Saved → frammer_charts/kpi06_publish_rate_by_channel.json


In [9]:
# ── Calculation ───────────────────────────────────────────────────────────────
user_contrib = (
    df_clean[df_clean["published_cnt"] > 0]
    .groupby(["Channel", "User"])["published_cnt"]
    .sum()
    .reset_index()
)

channel_total = user_contrib.groupby("Channel")["published_cnt"].sum().rename("channel_total")
user_contrib  = user_contrib.join(channel_total, on="Channel")
user_contrib["contribution_pct"] = (
    user_contrib["published_cnt"] / user_contrib["channel_total"] * 100
).round(2)

# Top 3 per channel
top3 = (
    user_contrib.groupby("Channel")
    .apply(lambda g: g.nlargest(3, "contribution_pct"))
    .reset_index(drop=True)
)

# ── Small Multiples Layout ────────────────────────────────────────────────────
channels = sorted(top3["Channel"].unique())
n_cols   = 4
n_rows   = int(np.ceil(len(channels) / n_cols))

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"Channel {ch}" for ch in channels],
    horizontal_spacing=0.08,
    vertical_spacing=0.15
)

for i, ch in enumerate(channels):
    row = i // n_cols + 1
    col = i %  n_cols + 1
    sub = top3[top3["Channel"] == ch].sort_values("contribution_pct", ascending=False)

    fig.add_trace(
        go.Bar(
            x=sub["User"],
            y=sub["contribution_pct"],
            marker_color=FRAMMER_COLORS[2],
            marker_line_color="#0D0D0D",
            marker_line_width=1,
            text=sub["contribution_pct"].apply(lambda x: f"{x:.0f}%"),
            textposition="outside",
            textfont=dict(color="#FFFFFF", size=9),
            hovertemplate="<b>%{x}</b><br>Contribution: %{y:.1f}%<extra></extra>",
            showlegend=False,
        ),
        row=row, col=col
    )

# Update all subplot axes
fig.update_xaxes(tickfont=dict(size=8, color="#FFFFFF"),
                 linecolor="#444444", gridcolor="#2A2A2A")
fig.update_yaxes(tickfont=dict(size=8, color="#FFFFFF"),
                 linecolor="#444444", gridcolor="#2A2A2A",
                 title_text="Contribution %")

fig.update_layout(
    **FRAMMER_LAYOUT,
    title="KPI-06 · Top 3 User Contribution % per Channel<br>"
          "<sup>Each user's share of channel's total published count</sup>",
    height=n_rows * 280,
    width=1100,
)

# Update subplot title colors
for annotation in fig.layout.annotations:
    annotation.font.color = "#FF3131"
    annotation.font.size  = 10

save_chart(fig, "kpi06_user_contribution_per_channel")

✅ Saved → frammer_charts/kpi06_user_contribution_per_channel.json


In [12]:
# ── Calculation ───────────────────────────────────────────────────────────────
kpi06_grouped = (
    df_clean.groupby("Channel")
    .agg(total_created  =("created_cnt",   "sum"),
         total_published=("published_cnt",  "sum"))
    .reset_index()
    .sort_values("total_created", ascending=False)
)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Bar(
    name="Created",
    x=kpi06_grouped["Channel"],
    y=kpi06_grouped["total_created"],
    marker_color=FRAMMER_COLORS[2],
    marker_line_color="#0D0D0D",
    marker_line_width=1.5,
    text=kpi06_grouped["total_created"],
    textposition="outside",
    textfont=dict(color="#FFFFFF", size=9),
    hovertemplate="<b>Channel %{x}</b><br>Created: %{y}<extra></extra>",
))

fig.add_trace(go.Bar(
    name="Published",
    x=kpi06_grouped["Channel"],
    y=kpi06_grouped["total_published"],
    marker_color=FRAMMER_COLORS[3],
    marker_line_color="#0D0D0D",
    marker_line_width=1.5,
    text=kpi06_grouped["total_published"],
    textposition="outside",
    textfont=dict(color="#FFFFFF", size=9),
    hovertemplate="<b>Channel %{x}</b><br>Published: %{y}<extra></extra>",
))

fig.update_layout(
    **FRAMMER_LAYOUT,
    title="KPI-06 · Created vs Published per Channel<br>",
    xaxis_title="Channel",
    yaxis_title="Count",
    barmode="group",
    legend=dict(
        font=dict(color="#FFFFFF"),
        bgcolor="#1A1A1A",
    ),
    height=500,
    width=900,
)

save_chart(fig, "kpi06_created_vs_published")

✅ Saved → frammer_charts/kpi06_created_vs_published.json


In [14]:
# ── Calculation ───────────────────────────────────────────────────────────────
top3_uploaders = (
    df_clean.groupby(["Channel", "User"])["uploaded_cnt"]
    .sum()
    .reset_index()
    .groupby("Channel")
    .apply(lambda g: g.nlargest(3, "uploaded_cnt"))
    .reset_index(drop=True)
)

# ── Small Multiples Layout ────────────────────────────────────────────────────
channels = sorted(top3_uploaders["Channel"].unique())
n_cols   = 4
n_rows   = int(np.ceil(len(channels) / n_cols))

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"Channel {ch}" for ch in channels],
    horizontal_spacing=0.08,
    vertical_spacing=0.15
)

for i, ch in enumerate(channels):
    row = i // n_cols + 1
    col = i %  n_cols + 1
    sub = top3_uploaders[top3_uploaders["Channel"] == ch].sort_values("uploaded_cnt", ascending=False)

    fig.add_trace(
        go.Bar(
            x=sub["User"],
            y=sub["uploaded_cnt"],
            marker_color=FRAMMER_COLORS[2],
            marker_line_color="#0D0D0D",
            marker_line_width=1,
            text=sub["uploaded_cnt"],
            textposition="outside",
            textfont=dict(color="#FFFFFF", size=9),
            hovertemplate="<b>%{x}</b><br>Uploads: %{y}<extra></extra>",
            showlegend=False,
        ),
        row=row, col=col
    )

# Update all subplot axes
fig.update_xaxes(tickfont=dict(size=8, color="#FFFFFF"),
                 linecolor="#444444", gridcolor="#2A2A2A")
fig.update_yaxes(tickfont=dict(size=8, color="#FFFFFF"),
                 linecolor="#444444", gridcolor="#2A2A2A",
                 title_text="Uploads")

fig.update_layout(
    **FRAMMER_LAYOUT,
    title="KPI-17 · Top 3 Uploaders per Channel<br>"
          "<sup>Users driving upload volume in each channel</sup>",
    height=n_rows * 280,
    width=1100,
)

# Subplot title colors
for annotation in fig.layout.annotations:
    annotation.font.color = "#FF3131"
    annotation.font.size  = 10

save_chart(fig, "kpi17_top3_uploaders_per_channel")

✅ Saved → frammer_charts/kpi17_top3_uploaders_per_channel.json


In [13]:
# ── Calculation ───────────────────────────────────────────────────────────────
kpi17 = (
    df_clean.groupby("Channel")["uploaded_cnt"]
    .sum()
    .reset_index()
    .rename(columns={"uploaded_cnt": "total_uploads"})
    .sort_values("total_uploads", ascending=False)
)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Bar(
    x=kpi17["Channel"],
    y=kpi17["total_uploads"],
    marker_color=FRAMMER_COLORS[2],
    marker_line_color="#0D0D0D",
    marker_line_width=1.5,
    text=kpi17["total_uploads"],
    textposition="outside",
    textfont=dict(color="#FFFFFF", size=9),
    hovertemplate="<b>Channel %{x}</b><br>Total Uploads: %{y}<extra></extra>",
))

fig.update_layout(
    **FRAMMER_LAYOUT,
    title="KPI-17 · Upload Volume per Channel<br>"
          "<sup>Total uploaded content per channel</sup>",
    xaxis_title="Channel",
    yaxis_title="Total Uploaded Count",
    height=500,
    width=900,
)

save_chart(fig, "kpi17_upload_volume_per_channel")

✅ Saved → frammer_charts/kpi17_upload_volume_per_channel.json


In [15]:
# ── Calculation ───────────────────────────────────────────────────────────────
kpi16 = (
    df_clean.groupby("User")
    .agg(total_published=("published_cnt", "sum"),
         total_created  =("created_cnt",   "sum"))
    .reset_index()
)

kpi16 = kpi16[kpi16["total_created"] >= 10].copy()

kpi16["user_publish_rate"] = (
    kpi16["total_published"] / kpi16["total_created"] * 100
).round(2)

top10 = kpi16.nlargest(10, "user_publish_rate").sort_values("user_publish_rate", ascending=True)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Bar(
    x=top10["user_publish_rate"],
    y=top10["User"],
    orientation="h",
    marker_color=FRAMMER_COLORS[2],
    marker_line_color="#0D0D0D",
    marker_line_width=1.5,
    text=top10["user_publish_rate"].apply(lambda x: f"{x:.1f}%"),
    textposition="outside",
    textfont=dict(color="#FFFFFF", size=9),
    hovertemplate="<b>%{y}</b><br>Publish Rate: %{x:.1f}%<br>"
                  "Published: %{customdata[0]}<br>"
                  "Created: %{customdata[1]}<extra></extra>",
    customdata=top10[["total_published", "total_created"]].values,
))

# 3% Benchmark line
fig.add_vline(
    x=3,
    line_dash="dash",
    line_color=FRAMMER_COLORS[4],
    line_width=1.5,
    annotation_text="3% Benchmark",
    annotation_font_color=FRAMMER_COLORS[4],
    annotation_position="top right"
)

fig.update_layout(
    **FRAMMER_LAYOUT,
    title="KPI-16 · Top 10 Users by Publish Rate %<br>"
          "<sup>Minimum 10 creations  |  Published ÷ Created × 100</sup>",
    xaxis_title="Publish Rate (%)",
    yaxis_title="User",
    height=520,
    width=900,
)

save_chart(fig, "kpi16_top10_users_publish_rate")

✅ Saved → frammer_charts/kpi16_top10_users_publish_rate.json


In [16]:
# ── Calculation ───────────────────────────────────────────────────────────────
kpi16_channel = (
    df_clean.groupby(["Channel", "User"])
    .agg(total_published=("published_cnt", "sum"),
         total_created  =("created_cnt",   "sum"))
    .reset_index()
)

kpi16_channel = kpi16_channel[kpi16_channel["total_created"] >= 5].copy()

kpi16_channel["user_publish_rate"] = (
    kpi16_channel["total_published"] / kpi16_channel["total_created"] * 100
).round(2)

# Top 3 users per channel by publish rate
top3_rate = (
    kpi16_channel.groupby("Channel")
    .apply(lambda g: g.nlargest(3, "user_publish_rate"))
    .reset_index(drop=True)
)

# ── Small Multiples Layout ────────────────────────────────────────────────────
channels = sorted(top3_rate["Channel"].unique())
n_cols   = 4
n_rows   = int(np.ceil(len(channels) / n_cols))

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"Channel {ch}" for ch in channels],
    horizontal_spacing=0.08,
    vertical_spacing=0.15
)

for i, ch in enumerate(channels):
    row = i // n_cols + 1
    col = i %  n_cols + 1
    sub = top3_rate[top3_rate["Channel"] == ch].sort_values("user_publish_rate", ascending=False)

    fig.add_trace(
        go.Bar(
            x=sub["User"],
            y=sub["user_publish_rate"],
            marker_color=FRAMMER_COLORS[2],
            marker_line_color="#0D0D0D",
            marker_line_width=1,
            text=sub["user_publish_rate"].apply(lambda x: f"{x:.1f}%"),
            textposition="outside",
            textfont=dict(color="#FFFFFF", size=9),
            hovertemplate="<b>%{x}</b><br>Publish Rate: %{y:.1f}%<extra></extra>",
            showlegend=False,
        ),
        row=row, col=col
    )

    # 3% benchmark per subplot
    fig.add_hline(
        y=3,
        line_dash="dash",
        line_color=FRAMMER_COLORS[4],
        line_width=1,
        row=row, col=col
    )

# Update all subplot axes
fig.update_xaxes(tickfont=dict(size=8, color="#FFFFFF"),
                 linecolor="#444444", gridcolor="#2A2A2A")
fig.update_yaxes(tickfont=dict(size=8, color="#FFFFFF"),
                 linecolor="#444444", gridcolor="#2A2A2A",
                 title_text="Publish Rate %")

fig.update_layout(
    **FRAMMER_LAYOUT,
    title="KPI-16 · Top 3 User Publish Rate % within each Channel<br>"
          "<sup>Min 5 creations  |  -- 3% Benchmark</sup>",
    height=n_rows * 280,
    width=1100,
)

# Subplot title colors
for annotation in fig.layout.annotations:
    annotation.font.color = "#FF3131"
    annotation.font.size  = 10

save_chart(fig, "kpi16_user_publish_rate_per_channel")

✅ Saved → frammer_charts/kpi16_user_publish_rate_per_channel.json


In [23]:
fig.update_layout(
    **FRAMMER_LAYOUT,
    title="KPI-18 · User × Channel Publish Rate % Heatmap<br>"
          "<sup>Same user behaving differently across channels</sup>",
    height=max(500, len(pivot) * 35),
    width=1000,
)

# Update axes separately
fig.update_xaxes(tickfont=dict(color="#FFFFFF", size=10),
                 linecolor="#444444", gridcolor="#2A2A2A",
                 title_text="Channel")
fig.update_yaxes(tickfont=dict(color="#FFFFFF", size=9),
                 linecolor="#444444", gridcolor="#2A2A2A",
                 title_text="User")

In [24]:
# ── Calculation ───────────────────────────────────────────────────────────────
kpi18_sm = (
    df_clean.groupby(["Channel", "User"])
    .agg(total_published=("published_cnt", "sum"),
         total_created  =("created_cnt",   "sum"))
    .reset_index()
)

kpi18_sm = kpi18_sm[kpi18_sm["total_created"] >= 5].copy()

kpi18_sm["user_publish_rate"] = (
    kpi18_sm["total_published"] / kpi18_sm["total_created"] * 100
).round(2)

# Top 8 users by total published count
top_users = (
    kpi18_sm.groupby("User")["total_published"]
    .sum()
    .nlargest(8)
    .index
)
kpi18_sm = kpi18_sm[kpi18_sm["User"].isin(top_users)]

# ── Small Multiples Layout ────────────────────────────────────────────────────
n_cols = 4
n_rows = int(np.ceil(len(top_users) / n_cols))

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=list(top_users),
    horizontal_spacing=0.08,
    vertical_spacing=0.15
)

for i, user in enumerate(top_users):
    row = i // n_cols + 1
    col = i %  n_cols + 1
    sub = (
        kpi18_sm[kpi18_sm["User"] == user]
        .sort_values("user_publish_rate", ascending=False)
    )

    fig.add_trace(
        go.Bar(
            x=sub["Channel"],
            y=sub["user_publish_rate"],
            marker_color=FRAMMER_COLORS[2],
            marker_line_color="#0D0D0D",
            marker_line_width=1,
            text=sub["user_publish_rate"].apply(lambda x: f"{x:.1f}%"),
            textposition="outside",
            textfont=dict(color="#FFFFFF", size=9),
            hovertemplate="<b>Channel %{x}</b><br>"
                          "Publish Rate: %{y:.1f}%<extra></extra>",
            showlegend=False,
        ),
        row=row, col=col
    )

    # 3% benchmark per subplot
    fig.add_hline(
        y=3,
        line_dash="dash",
        line_color=FRAMMER_COLORS[4],
        line_width=1,
        row=row, col=col
    )

# Update all subplot axes
fig.update_xaxes(tickfont=dict(size=8, color="#FFFFFF"),
                 linecolor="#444444", gridcolor="#2A2A2A",
                 title_text="Channel")
fig.update_yaxes(tickfont=dict(size=8, color="#FFFFFF"),
                 linecolor="#444444", gridcolor="#2A2A2A",
                 title_text="Publish Rate %")

fig.update_layout(
    **FRAMMER_LAYOUT,
    title="KPI-18 · Top Users — Publish Rate % across Channels<br>"
          "<sup>Each chart = one user  |  Bars = channels they appear in  |  -- 3% Benchmark</sup>",
    height=n_rows * 300,
    width=1100,
)

# Subplot title colors
for annotation in fig.layout.annotations:
    annotation.font.color = "#FF3131"
    annotation.font.size  = 10

save_chart(fig, "kpi18_small_multiples_top_users")

✅ Saved → frammer_charts/kpi18_small_multiples_top_users.json


In [25]:
# ── Pre-compute all KPIs ──────────────────────────────────────────────────────

# KPI-06
kpi06 = (
    df_clean.groupby("Channel")
    .agg(total_published=("published_cnt", "sum"),
         total_created  =("created_cnt",   "sum"))
    .assign(publish_rate_pct=lambda x:
            (x["total_published"] / x["total_created"] * 100).round(2))
    .reset_index()
    .sort_values("publish_rate_pct", ascending=False)
)

# KPI-17
kpi17 = (
    df_clean.groupby("Channel")["uploaded_cnt"]
    .sum().reset_index()
    .rename(columns={"uploaded_cnt": "total_uploads"})
    .sort_values("total_uploads", ascending=False)
)

top_uploader = (
    df_clean.groupby(["Channel", "User"])["uploaded_cnt"]
    .sum().reset_index()
    .sort_values("uploaded_cnt", ascending=False)
    .groupby("Channel").first().reset_index()
    .rename(columns={"uploaded_cnt": "top_uploads"})
    .sort_values("top_uploads", ascending=False)
)

# KPI-16
kpi16 = (
    df_clean.groupby("User")
    .agg(total_published=("published_cnt", "sum"),
         total_created  =("created_cnt",   "sum"))
    .reset_index()
)
kpi16 = kpi16[kpi16["total_created"] >= 10].copy()
kpi16["user_publish_rate"] = (
    kpi16["total_published"] / kpi16["total_created"] * 100
).round(2)
top10 = kpi16.nlargest(10, "user_publish_rate").sort_values("user_publish_rate", ascending=True)

# KPI-18
kpi18 = (
    df_clean.groupby(["Channel", "User"])
    .agg(total_published=("published_cnt", "sum"),
         total_created  =("created_cnt",   "sum"))
    .reset_index()
)
kpi18 = kpi18[kpi18["total_created"] >= 5].copy()
kpi18["user_publish_rate"] = (
    kpi18["total_published"] / kpi18["total_created"] * 100
).round(2)
pivot = kpi18.pivot_table(index="User", columns="Channel",
                           values="user_publish_rate", fill_value=np.nan)
pivot = pivot[pivot.notna().sum(axis=1) >= 2]

# ── Dashboard Layout ──────────────────────────────────────────────────────────
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        "KPI-06 · Publish Rate % per Channel",
        "KPI-06 · Created vs Published per Channel",
        "KPI-17 · Upload Volume per Channel",
        "KPI-17 · Top Uploader per Channel",
        "KPI-16 · Top 10 Users by Publish Rate %",
        "KPI-18 · User × Channel Heatmap",
    ],
    horizontal_spacing=0.12,
    vertical_spacing=0.12,
    specs=[
        [{"type": "bar"},     {"type": "bar"}],
        [{"type": "bar"},     {"type": "bar"}],
        [{"type": "bar"},     {"type": "heatmap"}],
    ]
)

# ── Panel 1 · KPI-06 Publish Rate % ──────────────────────────────────────────
fig.add_trace(go.Bar(
    x=kpi06["Channel"],
    y=kpi06["publish_rate_pct"],
    marker_color=FRAMMER_COLORS[2],
    marker_line_color="#0D0D0D",
    text=kpi06["publish_rate_pct"].apply(lambda x: f"{x:.1f}%"),
    textposition="outside",
    textfont=dict(color="#FFFFFF", size=8),
    hovertemplate="<b>Channel %{x}</b><br>Publish Rate: %{y:.1f}%<extra></extra>",
    showlegend=False,
), row=1, col=1)

fig.add_hline(y=3, line_dash="dash", line_color=FRAMMER_COLORS[4],
              line_width=1, row=1, col=1)

# ── Panel 2 · KPI-06 Created vs Published ────────────────────────────────────
fig.add_trace(go.Bar(
    name="Created",
    x=kpi06["Channel"],
    y=kpi06["total_created"],
    marker_color=FRAMMER_COLORS[1],
    marker_line_color="#0D0D0D",
    hovertemplate="<b>Channel %{x}</b><br>Created: %{y}<extra></extra>",
    showlegend=True,
), row=1, col=2)

fig.add_trace(go.Bar(
    name="Published",
    x=kpi06["Channel"],
    y=kpi06["total_published"],
    marker_color=FRAMMER_COLORS[3],
    marker_line_color="#0D0D0D",
    hovertemplate="<b>Channel %{x}</b><br>Published: %{y}<extra></extra>",
    showlegend=True,
), row=1, col=2)

# ── Panel 3 · KPI-17 Upload Volume ───────────────────────────────────────────
fig.add_trace(go.Bar(
    x=kpi17["Channel"],
    y=kpi17["total_uploads"],
    marker_color=FRAMMER_COLORS[2],
    marker_line_color="#0D0D0D",
    text=kpi17["total_uploads"],
    textposition="outside",
    textfont=dict(color="#FFFFFF", size=8),
    hovertemplate="<b>Channel %{x}</b><br>Uploads: %{y}<extra></extra>",
    showlegend=False,
), row=2, col=1)

# ── Panel 4 · KPI-17 Top Uploader ────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=top_uploader["Channel"],
    y=top_uploader["top_uploads"],
    marker_color=FRAMMER_COLORS[1],
    marker_line_color="#0D0D0D",
    text=top_uploader["User"],
    textposition="outside",
    textfont=dict(color="#FFD6D6", size=7),
    hovertemplate="<b>Channel %{x}</b><br>Top User: %{text}<br>Uploads: %{y}<extra></extra>",
    showlegend=False,
), row=2, col=2)

# ── Panel 5 · KPI-16 Top 10 Users ────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=top10["user_publish_rate"],
    y=top10["User"],
    orientation="h",
    marker_color=FRAMMER_COLORS[2],
    marker_line_color="#0D0D0D",
    text=top10["user_publish_rate"].apply(lambda x: f"{x:.1f}%"),
    textposition="outside",
    textfont=dict(color="#FFFFFF", size=8),
    hovertemplate="<b>%{y}</b><br>Publish Rate: %{x:.1f}%<extra></extra>",
    showlegend=False,
), row=3, col=1)

fig.add_vline(x=3, line_dash="dash", line_color=FRAMMER_COLORS[4],
              line_width=1, row=3, col=1)

# ── Panel 6 · KPI-18 Heatmap ─────────────────────────────────────────────────
fig.add_trace(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale=[
        [0.0,  "#7A0000"],
        [0.2,  "#CC0000"],
        [0.4,  "#FF3131"],
        [0.6,  "#FF9A9A"],
        [0.8,  "#FFD6D6"],
        [1.0,  "#FFFFFF"],
    ],
    texttemplate="%{z:.0f}%",
    textfont=dict(size=7),
    hovertemplate="<b>User:</b> %{y}<br>"
                  "<b>Channel:</b> %{x}<br>"
                  "<b>Publish Rate:</b> %{z:.1f}%<extra></extra>",
    showscale=False,
), row=3, col=2)

# ── Global Layout ─────────────────────────────────────────────────────────────
fig.update_layout(
    **FRAMMER_LAYOUT,
    title=dict(
        text="Frammer AI — Product Usage Analytics Dashboard",
        font=dict(size=18, color="#FF3131"),
        x=0.5
    ),
    barmode="group",
    height=1200,
    width=1300,
    legend=dict(
        font=dict(color="#FFFFFF"),
        bgcolor="#1A1A1A",
    ),
)

fig.update_xaxes(tickfont=dict(color="#FFFFFF", size=8),
                 linecolor="#444444", gridcolor="#2A2A2A")
fig.update_yaxes(tickfont=dict(color="#FFFFFF", size=8),
                 linecolor="#444444", gridcolor="#2A2A2A")

# Subplot title colors
for annotation in fig.layout.annotations:
    annotation.font.color = "#FF3131"
    annotation.font.size  = 10

save_chart(fig, "dashboard_summary")

✅ Saved → frammer_charts/dashboard_summary.json
